# Clinical Trials Data Lakehouse Pipeline
## Complete Architecture & Pipeline Overview

---

### Project Overview

This data lakehouse implements a **Medallion Architecture** (Bronze → Silver → Gold) to process clinical trial data from raw CSV files into analytics-ready dimensional models. The pipeline handles **500K+ clinical trial records** with automated data quality checks, standardization, and star schema modeling.

**Technology Stack:**
* **Storage:** AWS S3 + Delta Lake
* **Processing:** Apache Spark (Databricks)
* **Architecture:** Medallion (Bronze/Silver/Gold)
* **Catalog:** Unity Catalog
* **Orchestration:** Databricks Jobs

---

## Dataset Overview

### Source Data: ClinicalTrials.gov

The pipeline processes clinical trial studies data exported from ClinicalTrials.gov, the world's largest registry of clinical studies.

**Dataset Characteristics:**
* **Format:** CSV files (multi-line, quoted fields)
* **Size:** 500K+ trial records across multiple files
* **Location:** AWS S3 (`s3://clinical-trial4/staging/`)
* **Update Frequency:** Batch loads (incremental capable)

**Key Attributes (28 fields):**

| Category | Fields |
| --- | --- |
| **Identifiers** | NCT Number, Study Title, Acronym, Other IDs |
| **Study Info** | Study Status, Study Type, Phases, Study Design |
| **Medical** | Conditions, Interventions, Primary/Secondary Outcomes |
| **Participants** | Sex, Age Groups, Enrollment |
| **Organizations** | Sponsor, Collaborators, Funder Type, Locations |
| **Dates** | Start Date, Completion Date, First Posted, Last Update |
| **Results** | Study Results, Brief Summary |

**Data Quality Challenges:**
* Inconsistent date formats
* Missing values in enrollment and dates
* Non-standard NCT number formats
* Multi-line text fields with special characters
* Duplicate records across batches

---

## Medallion Architecture

### The Three-Layer Approach

The pipeline follows Databricks' **Medallion Architecture** pattern for progressive data refinement:

```
┌─────────────────┐       ┌─────────────────┐       ┌─────────────────┐
│                 │       │                 │       │                 │
│  BRONZE        │ ───▶  │  SILVER         │ ───▶  │  GOLD           │
│  Raw Landing    │       │  Curated        │       │  Analytics      │
│                 │       │                 │       │                 │
└─────────────────┘       └─────────────────┘       └─────────────────┘
   • CSV Ingestion          • Data Cleaning          • Star Schema
   • Schema Enforcement      • Validation             • Fact Tables
   • Audit Columns           • Standardization        • Dimensions
   • Minimal Transform       • Deduplication          • Aggregations
```

### Layer Responsibilities

**Bronze Layer (Raw Zone)**
* Ingest raw CSV files "as-is" from S3
* Apply explicit schema for type safety
* Add audit metadata (timestamps, versions, source files)
* Preserve original data for reprocessing
* **No business logic** applied

**Silver Layer (Curated Zone)**
* Clean and standardize data
* Validate patterns (NCT numbers, dates)
* Handle NULL values
* Remove duplicates
* Apply business transformations
* Filter invalid records to rejection tables

**Gold Layer (Analytics Zone)**
* Build star schema (fact + dimension tables)
* Create surrogate keys for dimensions
* Perform SCD Type 1 updates
* Optimize for analytics queries
* Ready for BI dashboards and reports

---

## Bronze Layer Details

### Notebook: `01_Bronze_Load`

**Purpose:** Raw data ingestion from S3 staging area into Delta Lake

**Input:**
* CSV files from `s3://clinical-trial4/staging/part1/` (parameterized)
* Multi-line CSVs with header row

**Processing Steps:**

1. **Read CSV with Explicit Schema**
   * 28 fields defined (StringType, DateType, IntegerType)
   * Date columns: Start Date, Completion Date, First Posted, etc.
   * Numeric: Enrollment (IntegerType)
   * All other fields: StringType

2. **Add Audit Columns**
   * `bronze_load_timestamp` - When record was loaded
   * `batch_id` - Batch identifier (yyyyMMddHHmmss format)
   * `source_file` - Original CSV filename
   * `version` - Incremental version number

3. **Write to Delta Table**
   * Table: `clinical_datalakehouse_modernization.bronze.clinical_trial_studies_bronze`
   * Location: `s3://clinical-trial4/bronze/clinical_trial_studies/`
   * Mode: Append (preserves history)

**Data Quality:**
* Row count validation (before/after)
* Schema verification
* Logging for observability

**Output:**
* Bronze Delta table with ~521K records per full load
* Original data structure preserved
* Audit trail for lineage tracking

---

## Silver Layer Details

### Notebook: `02_Silver_Load`

**Purpose:** Data cleaning, standardization, and validation

**Input:**
* Bronze table: `clinical_datalakehouse_modernization.bronze.clinical_trial_studies_bronze`
* Latest batch (incremental processing capable)

**Processing Steps:**

### 1. Column Standardization
* Convert column names to lowercase
* Replace spaces/hyphens/slashes with underscores
* Example: `"NCT Number"` → `"nct_number"`

### 2. String Data Cleaning
* Trim whitespace from all string columns
* Replace multiple spaces with single space
* Convert empty strings to NULL
* Remove leading/trailing special characters

### 3. NULL Value Handling
* `enrollment`: NULL → 0
* `study_duration_days`: NULL → 0
* `first_posted_year`: NULL → 9999 (Unknown)
* `first_posted_month`: NULL → 0 (Unknown)

### 4. Pattern Validation
* **NCT Number Format:** Must match `^NCT[0-9]{8}$`
* **Valid records:** 195K+ (37%)
* **Rejected records:** 325K+ (63%) - moved to rejection table
* Rejection reason logged for analysis

### 5. Feature Engineering
* **study_duration_days:** DATEDIFF(completion_date, start_date)
* **enrollment_bucket:** SMALL (<100), MEDIUM (<1000), LARGE (>=1000)
* **has_child:** Flag if age contains "CHILD"
* **has_adult:** Flag if age contains "ADULT"
* **has_older_adult:** Flag if age contains "OLDER_ADULT"
* **first_posted_year:** YEAR(first_posted) with NULL handling
* **first_posted_month:** MONTH(first_posted) with NULL handling
* **status_category:** Grouped status (ACTIVE, COMPLETED, STOPPED, OTHER)

### 6. Deduplication
* Identify duplicates based on NCT Number
* Keep latest record (by Last Update Posted)
* Log duplicates to separate table for audit

**Output Tables:**
* `clinical_datalakehouse_modernization.silver.clinical_trial_studies_silver` (~195K valid records)
* `clinical_datalakehouse_modernization.silver.clinical_trial_rejected_records` (invalid format)
* `clinical_datalakehouse_modernization.silver.clinical_trial_duplicate_log` (duplicates)

**Key Transformations:**
* 62% rejection rate due to invalid NCT formats
* Date handling prevents NULL propagation to Gold layer
* Status categorization simplifies downstream analysis

---

## Gold Layer Details

### Notebook: `03_Gold_Load`

**Purpose:** Build analytics-ready star schema for BI and reporting

**Input:**
* Silver table: `clinical_datalakehouse_modernization.silver.clinical_trial_studies_silver`

**Architecture: Star Schema**

```
                    ┌──────────────────┐
                    │                  │
                    │   dim_status     │
                    │                  │
                    └────────┬─────────┘
                             │
        ┌────────────────────┼────────────────────┐
        │                    │                    │
  ┌─────▼─────┐       ┌─────▼─────┐       ┌─────▼─────┐
  │           │       │           │       │           │
  │dim_phase  │       │FACT TABLE │       │dim_sponsor│
  │           │       │           │       │           │
  └───────────┘       └─────┬─────┘       └───────────┘
                            │
        ┌───────────────────┼───────────────────┐
        │                   │                   │
  ┌─────▼─────┐      ┌─────▼─────┐      ┌─────▼─────┐
  │           │      │           │      │           │
  │dim_funder │      │dim_location│     │   (more)  │
  │           │      │           │      │           │
  └───────────┘      └───────────┘      └───────────┘
```

### Dimension Tables (SCD Type 1)

**1. dim_status**
* Surrogate Key: `status_key`
* Attributes: `study_status`, `status_category`
* Example: "RECRUITING" → "ACTIVE"

**2. dim_phase**
* Surrogate Key: `phase_key`
* Attributes: `phase`
* Values: PHASE1, PHASE2, PHASE3, PHASE4, NA, etc.

**3. dim_sponsor**
* Surrogate Key: `sponsor_key`
* Attributes: `sponsor_name`
* Handles NULL with "UNKNOWN"

**4. dim_location**
* Surrogate Key: `location_key`
* Attributes: `location_name`
* Multi-value field (some trials have multiple locations)

**5. dim_funder_type**
* Surrogate Key: `funder_type_key`
* Attributes: `funder_type`
* Values: INDUSTRY, NIH, OTHER, FED, etc.

### Fact Table: fact_clinical_trials

**Grain:** One row per clinical trial (NCT Number)

**Foreign Keys:**
* `status_key` → dim_status
* `phase_key` → dim_phase
* `sponsor_key` → dim_sponsor
* `location_key` → dim_location
* `funder_type_key` → dim_funder_type

**Measures (Numeric Facts):**
* `enrollment` - Number of participants
* `has_child`, `has_adult`, `has_older_adult` - Age group flags

**Degenerate Dimensions:**
* `trial_id` (NCT Number) - Business key
* `first_posted_year`, `first_posted_month` - Time dimensions

**Date Handling:**
* NULL years → 9999 (Unknown Year)
* NULL months → 0 (Unknown Month)
* Allows time-series analysis without breaking on NULLs

**Load Strategy:**
* Dimensions: INSERT if new, UPDATE if changed (SCD Type 1)
* Fact: Full refresh or incremental merge based on NCT Number
* Surrogate keys generated using ROW_NUMBER() or monotonically_increasing_id()

**Output Tables:**
* `clinical_datalakehouse_modernization.gold.fact_clinical_trials`
* `clinical_datalakehouse_modernization.gold.dim_status`
* `clinical_datalakehouse_modernization.gold.dim_phase`
* `clinical_datalakehouse_modernization.gold.dim_sponsor`
* `clinical_datalakehouse_modernization.gold.dim_location`
* `clinical_datalakehouse_modernization.gold.dim_funder_type`

**Analytics Benefits:**
* Optimized for aggregations and drill-downs
* Consistent surrogate keys for joins
* Simplified querying (no complex transformations needed)
* Dashboard-ready data model

---

## Pipeline Orchestration & Workflow

### Orchestration Notebooks

**1. `04_Audit_And_Orchestration`**
* Sequential execution: Bronze → Silver → Gold
* Uses `dbutils.notebook.run()` to call each layer
* Captures execution times and record counts
* Logs success/failure for each stage

**2. `05_Truncate_Load_Control`**
* Full refresh capability (truncate + reload)
* Deletes existing data from all layers
* Triggers complete pipeline rerun
* Used for testing or data fixes

### Data Flow Sequence

```
1. START
   ↓
2. [Bronze Notebook]
   • Read CSV from S3
   • Apply schema
   • Add audit columns
   • Write to Bronze Delta table
   ↓
3. [Silver Notebook]
   • Read Bronze table
   • Clean & standardize
   • Validate patterns
   • Handle NULLs
   • Filter invalid records
   • Write to Silver Delta table
   ↓
4. [Gold Notebook]
   • Read Silver table
   • Build dimensions (UPSERT)
   • Populate fact table with surrogate keys
   • Write to Gold Delta tables
   ↓
5. END (Analytics-ready star schema)
```

### Error Handling

* **Bronze:** Schema enforcement catches type mismatches
* **Silver:** Rejection tables capture invalid records
* **Gold:** Dimension lookups handle missing values with "UNKNOWN"
* **Logging:** Every layer logs record counts and timestamps

### Incremental Processing (Future Enhancement)

Currently: Full refresh per batch  
Planned:
* Bronze: Append new versions only
* Silver: Process only new Bronze versions
* Gold: MERGE statements to update changed records

---

## Key Metrics & Data Quality

### Volume Statistics

| Layer | Records | Status |
| --- | --- | --- |
| Bronze | 521,256 | Raw ingestion |
| Silver | 195,785 | 37.5% valid records |
| Gold Fact | 195,785 | Star schema |
| Rejected | 325,471 | 62.5% invalid NCT format |

### Data Quality Checks

**Validation Rules:**
* NCT Number must match `^NCT[0-9]{8}%md
## Key Metrics & Data Quality

### Volume Statistics

| Layer | Records | Status |
| --- | --- | --- |
| Bronze | 521,256 | Raw ingestion |
| Silver | 195,785 | 37.5% valid records |
| Gold Fact | 195,785 | Star schema |
| Rejected | 325,471 | 62.5% invalid NCT format |

### Data Quality Checks

**Validation Rules:**

* Date columns converted to proper DateType
* Enrollment must be numeric (0 for NULL)
* Deduplicated on NCT Number + Last Update

**NULL Handling Strategy:**
* Enrollment: 0 (no participants)
* Study Duration: 0 (unknown duration)
* First Posted Year: 9999 (unknown year)
* First Posted Month: 0 (unknown month)
* Text fields: NULL or "UNKNOWN" in dimensions

### Performance Considerations

**Storage:**
* Format: Delta Lake (ACID transactions, time travel)
* Compression: Parquet with Snappy
* Partitioning: None currently (can add by year/phase)

**Optimization Opportunities:**
* Add Z-ORDER clustering on frequently filtered columns
* Partition fact table by `first_posted_year`
* Enable Auto Optimize for small files
* Add bloom filters on trial_id for point lookups

---

## Dashboard & Analytics Usage

### Sample Queries for Dashboards

**1. Trials by Status Category**
```sql
SELECT 
    ds.status_category,
    COUNT(*) as trial_count,
    SUM(f.enrollment) as total_enrollment
FROM clinical_datalakehouse_modernization.gold.fact_clinical_trials f
JOIN clinical_datalakehouse_modernization.gold.dim_status ds 
    ON f.status_key = ds.status_key
GROUP BY ds.status_category
ORDER BY trial_count DESC
```

**2. Trials Over Time (Handling Unknown Dates)**
```sql
SELECT 
    CASE WHEN first_posted_year = 9999 THEN 'Unknown' 
         ELSE CAST(first_posted_year AS STRING) END as year,
    COUNT(*) as trial_count
FROM clinical_datalakehouse_modernization.gold.fact_clinical_trials
WHERE first_posted_year != 9999  -- Exclude unknown dates
GROUP BY first_posted_year
ORDER BY first_posted_year
```

**3. Top Sponsors by Trial Volume**
```sql
SELECT 
    sp.sponsor_name,
    COUNT(*) as trial_count,
    AVG(f.enrollment) as avg_enrollment
FROM clinical_datalakehouse_modernization.gold.fact_clinical_trials f
JOIN clinical_datalakehouse_modernization.gold.dim_sponsor sp 
    ON f.sponsor_key = sp.sponsor_key
WHERE sp.sponsor_name != 'UNKNOWN'
GROUP BY sp.sponsor_name
ORDER BY trial_count DESC
LIMIT 20
```

**4. Phase Distribution**
```sql
SELECT 
    dp.phase,
    COUNT(*) as trial_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
FROM clinical_datalakehouse_modernization.gold.fact_clinical_trials f
JOIN clinical_datalakehouse_modernization.gold.dim_phase dp 
    ON f.phase_key = dp.phase_key
GROUP BY dp.phase
ORDER BY trial_count DESC
```

**5. Age Group Analysis**
```sql
SELECT 
    SUM(has_child) as child_trials,
    SUM(has_adult) as adult_trials,
    SUM(has_older_adult) as older_adult_trials,
    COUNT(*) as total_trials
FROM clinical_datalakehouse_modernization.gold.fact_clinical_trials
```

### Handling NULL Date Values in Dashboards

**Option 1: Exclude Unknown Dates**
```sql
WHERE first_posted_year != 9999 AND first_posted_month != 0
```

**Option 2: Show as "Unknown" Category**
```sql
CASE WHEN first_posted_year = 9999 THEN 'Unknown' 
     ELSE CAST(first_posted_year AS STRING) END
```

**Option 3: Count Unknown Separately**
```sql
SELECT 
    CASE WHEN first_posted_year = 9999 THEN 'Unknown' ELSE 'Known' END as date_status,
    COUNT(*) as trial_count
FROM clinical_datalakehouse_modernization.gold.fact_clinical_trials
GROUP BY date_status
```

---

## Summary

### Pipeline Architecture

* **Bronze Layer:** Raw data ingestion with schema enforcement
* **Silver Layer:** Data cleaning, validation, and feature engineering
* **Gold Layer:** Star schema with fact and dimension tables
* **Orchestration:** Automated pipeline execution with audit logging  

### Key Features

* **Data Quality:** 37.5% valid records after NCT pattern validation
* **NULL Handling:** Default values prevent dashboard breaks (9999 for year, 0 for month)
* **Star Schema:** Optimized for analytics with 5 dimension tables
* **Audit Trail:** Timestamps, versions, and source tracking throughout
* **Scalability:** Delta Lake format enables ACID transactions and time travel

### Data Flow Summary

```
S3 CSV Files (521K records)
    ↓
Bronze Delta Table (521K records)
    ↓
Silver Delta Table (196K valid + 325K rejected)
    ↓
Gold Star Schema (196K facts + 5 dimensions)
    ↓
BI Dashboards & Analytics
```

### Next Steps & Enhancements

* **Incremental Processing:** Change Data Capture for efficiency
* **Partitioning:** Add date-based partitioning to fact table
* **Optimization:** Enable Z-ORDER and Auto Optimize
* **Data Quality:** Improve NCT validation or source data quality
* **Date Dimension:** Create proper calendar dimension table
* **Monitoring:** Add data quality dashboards and alerts
* **Security:** Implement row-level security and ABAC policies

---

### Related Notebooks

* [01_Bronze_Load](#notebook-931440776100916) - Raw data ingestion
* [02_Silver_Load](#notebook-766498940122026) - Data cleaning & validation
* [03_Gold_Load](#notebook-2408597408717905) - Star schema creation
* [04_Audit_And_Orchestration](#notebook-2427342087839636) - Pipeline orchestration
* [05_Truncate_Load_Control](#notebook-2427342087839637) - Full refresh control

---

**Documentation Version:** 1.0  
**Last Updated:** June 3, 2026  
**Author:** Sucharita Gopi